In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

D1 = Path("../data/d1/matches")

def tiros_de_partido(ruta):
    d = json.load(open(ruta, encoding="utf-8"))
    g = d["general"]
    shots = (d["content"].get("shotmap") or {}).get("shots", [])
    filas = []
    for s in shots:
        filas.append({
            "fotmob_id": int(g["matchId"]),
            "home_id": g["homeTeam"]["id"], "home": g["homeTeam"]["name"],
            "away_id": g["awayTeam"]["id"], "away": g["awayTeam"]["name"],
            "team_id": s.get("teamId"),
            "min": s.get("min"), "min_added": s.get("minAdded"), "period": s.get("period"),
            "evento": s.get("eventType"), "xg": s.get("expectedGoals"),
            "al_arco": s.get("isOnTarget"), "autogol": s.get("isOwnGoal"),
        })
    return pd.DataFrame(filas)

tiros_de_partido(sorted(D1.glob("*.json"))[0]).head()

: 

In [2]:
tiros = pd.concat([tiros_de_partido(f) for f in sorted(D1.glob("*.json"))],
                  ignore_index=True)
print(tiros.shape)
print(tiros.fotmob_id.nunique(), "partidos con tiros")
print(tiros.period.value_counts())
print(tiros.evento.value_counts())
print("tiros sin xG:", tiros.xg.isna().sum())

(2630, 13)
104 partidos con tiros
period
SecondHalf         1368
FirstHalf          1137
SecondHalfExtra      44
FirstHalfExtra       41
PenaltyShootout      40
Name: count, dtype: int64
evento
AttemptSaved    1253
Miss             994
Goal             333
Post              50
Name: count, dtype: int64
tiros sin xG: 14


In [3]:
# 1. Rango de minutos por periodo
print(tiros.groupby("period")["min"].agg(["min", "max"]))

# 2. ¿Los tiros sin xG son autogoles?
print(tiros[tiros.xg.isna()][["evento", "autogol", "period"]].value_counts())

# 3. Goles sin contar la tanda de penales (esperamos 308)
goles = tiros[(tiros.evento == "Goal") & (tiros.period != "PenaltyShootout")]
print("goles sin tanda:", len(goles))
print("autogoles:", goles.autogol.sum())

                 min  max
period                   
FirstHalf          1   45
FirstHalfExtra    91  105
PenaltyShootout  121  132
SecondHalf        46   90
SecondHalfExtra  106  120
evento  autogol  period         
Goal    True     SecondHalf         8
                 FirstHalf          5
                 SecondHalfExtra    1
Name: count, dtype: int64
goles sin tanda: 308
autogoles: 14


In [8]:
CHECKPOINTS = [15, 30, 45, 60, 75]

t90 = tiros[tiros.period.isin(["FirstHalf", "SecondHalf"])].copy()
t90["xg"] = t90["xg"].fillna(0)
t90["lado"] = np.where(t90.team_id == t90.home_id, "A", "B")

def hasta(df, t):
    """Tiros ocurridos hasta el checkpoint t."""
    if t < 45:
        return df[(df.period == "FirstHalf") & (df["min"] <= t)]
    if t == 45:
        return df[df.period == "FirstHalf"]          # incluye el 45+x
    return df[(df.period == "FirstHalf") |
              ((df.period == "SecondHalf") & (df["min"] <= t))]

filas = []
for mid, g in t90.groupby("fotmob_id"):
    for t in CHECKPOINTS:
        h = hasta(g, t)
        intentos = h[~h.autogol]          # un autogol no es un tiro del equipo
        fila = {"fotmob_id": mid, "t": t}
        for lado in ["A", "B"]:
            s = intentos[intentos.lado == lado]
            fila[f"xg_{lado}"] = s.xg.sum()
            fila[f"tiros_{lado}"] = len(s)
            fila[f"al_arco_{lado}"] = int(s.al_arco.sum())
        filas.append(fila)

estado = pd.DataFrame(filas)
print(estado.shape)          # esperamos (520, 8)
estado.head(10)


(520, 8)


,fotmob_id,t,xg_A,tiros_A,al_arco_A,xg_B,tiros_B,al_arco_B
0,4653703,15,0.071091,2,0,0.120171,2,2
1,4653703,30,0.071091,2,0,0.120171,2,2
2,4653703,45,0.146109,5,1,0.194596,3,3
3,4653703,60,0.425402,8,4,0.297227,5,4
4,4653703,75,0.516076,9,5,0.330636,6,4
5,4653704,15,0.015439,1,1,0.031389,1,1
6,4653704,30,0.134300,5,4,0.170221,2,1
7,4653704,45,0.947844,15,8,0.281060,3,1
8,4653704,60,1.594651,17,9,0.317473,4,1
9,4653704,75,2.648128,22,14,0.317473,4,1


In [9]:
estado.groupby("t")[["xg_A", "xg_B", "tiros_A", "tiros_B"]].mean().round(2)

,xg_A,xg_B,tiros_A,tiros_B
t,,,,
15,0.20,0.15,1.87,1.39
30,0.34,0.30,3.22,2.54
45,0.64,0.50,6.16,4.72
60,0.90,0.65,8.29,6.42
75,1.11,0.79,10.08,7.71


In [10]:
estado.shape

(520, 8)

Revisar autogoles

In [12]:
mapa = pd.read_csv("../data/processed/mapa_d1_d2.csv")
oficial = pd.read_csv("../data/d2/matches.csv")[["match_id", "home_score", "away_score"]]

# todos los goles menos la tanda de penales (el marcador oficial incluye la prórroga)
goles = tiros[(tiros.evento == "Goal") & (tiros.period != "PenaltyShootout")].copy()
ids = tiros.fotmob_id.unique()      # los 104 partidos, incluidos los 0-0

def lado_gol(goles, autogol_al_rival):
    lado = np.where(goles.team_id == goles.home_id, "A", "B")
    if autogol_al_rival:
        lado = np.where(goles.autogol, np.where(lado == "A", "B", "A"), lado)
    return lado

for regla in [True, False]:
    g = goles.assign(lado=lado_gol(goles, regla))
    tot = (g.pivot_table(index="fotmob_id", columns="lado", aggfunc="size", fill_value=0)
             .reindex(ids, fill_value=0).rename_axis("fotmob_id").reset_index())
    cmp = tot.merge(mapa, on="fotmob_id").merge(oficial, on="match_id")
    ok = ((cmp.A == cmp.home_score) & (cmp.B == cmp.away_score)).sum()
    print("autogol se suma al rival:" if regla else "autogol se suma al equipo registrado:", ok, "/ 104")

autogol se suma al rival: 104 / 104
autogol se suma al equipo registrado: 90 / 104


In [13]:
goles["lado"] = lado_gol(goles, autogol_al_rival=True)
g90 = goles[goles.period.isin(["FirstHalf", "SecondHalf"])]   # sin prórroga

# goles acumulados en cada checkpoint
filas = []
for mid in ids:
    g = g90[g90.fotmob_id == mid]
    for t in CHECKPOINTS:
        h = hasta(g, t)
        filas.append({"fotmob_id": mid, "t": t,
                      "goles_A": int((h.lado == "A").sum()),
                      "goles_B": int((h.lado == "B").sum())})
goles_t = pd.DataFrame(filas)

# etiqueta: resultado a los 90'  (0 = gana A, 1 = empate, 2 = gana B)
fin = (g90.groupby(["fotmob_id", "lado"]).size().unstack(fill_value=0)
          .reindex(ids, fill_value=0).rename_axis("fotmob_id"))
dif = fin["A"] - fin["B"]
etiqueta = pd.DataFrame({"fotmob_id": fin.index,
                         "y": np.select([dif > 0, dif == 0], [0, 1], 2)})

estado = estado.merge(goles_t, on=["fotmob_id", "t"]).merge(etiqueta, on="fotmob_id")
print(estado.shape)    # esperamos (520, 11)

(520, 11)


In [14]:
# distribución de resultados a los 90' (una fila por partido)
print(estado.drop_duplicates("fotmob_id").y.map({0: "gana A", 1: "empate", 2: "gana B"}).value_counts())

# los goles nunca pueden bajar entre checkpoints
print("¿goles que bajan?:", (estado.groupby("fotmob_id")[["goles_A", "goles_B"]].diff() < 0).any().any())

y
gana A    46
empate    29
gana B    29
Name: count, dtype: int64
¿goles que bajan?: False


In [15]:
Path("../data/processed").mkdir(parents=True, exist_ok=True)
estado.to_csv("../data/processed/estado_d1.csv", index=False)